# Week 6 - Spatial fire localisation training

Notebook huấn luyện phiên bản `spatial_heatmap_v2`. Chạy từng cell theo thứ tự từ trên xuống.

Pipeline: split sạch -> augmentation đồng bộ nhãn -> spatial heatmap 14x14 -> masked coordinate loss -> MAE/PCK.

In [ ]:
from pathlib import Path
from types import SimpleNamespace
import json
import sys
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

# Kaggle: ba đường dẫn dưới đây phải khớp với tên dataset bạn đã Add.
KAGGLE_CODE_DIR = Path('/kaggle/input/sam-experiment-code')
KAGGLE_MODEL_DIR = Path('/kaggle/input/fire-model-data')
KAGGLE_IMAGES_DIR = Path('/kaggle/input/fire-detection-from-cctv')
LOCAL_ROOT = Path(r'D:/LAB/SAM_Experiment')
IS_KAGGLE = Path('/kaggle/input').exists()
if IS_KAGGLE:
    ROOT = KAGGLE_MODEL_DIR
    CODE_ROOT = KAGGLE_CODE_DIR
    DATASET_ROOT = KAGGLE_IMAGES_DIR
    OUTPUT_DIR = Path('/kaggle/working/week6_spatial')
else:
    ROOT = LOCAL_ROOT
    CODE_ROOT = LOCAL_ROOT
    DATASET_ROOT = LOCAL_ROOT / 'fire-detection-from-cctv'
    OUTPUT_DIR = LOCAL_ROOT / 'fire-model-data' / 'week6_spatial'
if not CODE_ROOT.exists():
    raise FileNotFoundError(f'Không tìm thấy code root: {CODE_ROOT}. Hãy upload train_week6.py vào Kaggle dataset này.')
if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))

from train_week6 import (
    ARCHITECTURE, FireDataset, FireLoss, SpatialFireModel,
    load_records, run_epoch, save_checkpoint, seed_everything, split_records
)

SEED = 42
EPOCHS = 30
BATCH_SIZE = 32
NUM_WORKERS = 2 if torch.cuda.is_available() else 0
LEARNING_RATE = 1e-4
USE_PRETRAINED = True
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
LABEL_CANDIDATES = [
    ROOT / 'dataset_labels (1).json',
    ROOT / 'fire-model-data' / 'dataset_labels (1).json',
]
LABELS = next((path for path in LABEL_CANDIDATES if path.exists()), LABEL_CANDIDATES[0])
TRAIN_ARGS = SimpleNamespace(epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LEARNING_RATE)
seed_everything(SEED)
print('ROOT:', ROOT)
print('DEVICE:', DEVICE)
print('ARCHITECTURE:', ARCHITECTURE)
print('IS_KAGGLE:', IS_KAGGLE)
print('CODE_ROOT:', CODE_ROOT)
print('LABELS:', LABELS, 'exists:', LABELS.exists())
print('DATASET_ROOT:', DATASET_ROOT, 'exists:', DATASET_ROOT.exists())
if not LABELS.exists():
    raise FileNotFoundError(f'Không tìm thấy labels: {LABELS}')
if not DATASET_ROOT.exists():
    raise FileNotFoundError(f'Không tìm thấy dataset root: {DATASET_ROOT}')

## 1. Nạp dữ liệu và chia tập

Nếu metadata có `train/test`, test gốc được giữ nguyên. Chỉ train gốc được tách tiếp thành train/validation.

In [ ]:
if not LABELS.exists():
    raise FileNotFoundError(f'Không tìm thấy labels: {LABELS}')
if not DATASET_ROOT.exists():
    raise FileNotFoundError(f'Không tìm thấy dataset root: {DATASET_ROOT}')
records, load_stats = load_records(LABELS, DATASET_ROOT)
splits = split_records(records, seed=SEED)
split_info = {
    name: {
        'count': len(items),
        'fire': sum(int(item.has_fire) for item in items),
        'no_fire': sum(int(not item.has_fire) for item in items),
    }
    for name, items in splits.items()
}
print('load_stats:', load_stats)
print('split_info:', split_info)
assert all(split_info[name]['count'] > 0 for name in ('train', 'val', 'test'))

In [ ]:
train_loader = DataLoader(FireDataset(splits['train'], train=True), batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(FireDataset(splits['val'], train=False), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(FireDataset(splits['test'], train=False), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
images, targets, sizes, paths = next(iter(train_loader))
print('images:', tuple(images.shape))
print('targets:', tuple(targets.shape))
print('sizes:', tuple(sizes.shape))
print('first target [has_fire, x_norm, y_norm]:', targets[0].tolist())

## 2. Smoke test

Kiểm tra feature map, loss và backward mà không tải pretrained weights.

In [ ]:
smoke_model = SpatialFireModel(pretrained=False).to(DEVICE)
smoke_criterion = FireLoss(lambda_coord=5.0, lambda_heatmap=0.5)
smoke_outputs = smoke_model(images[:2].to(DEVICE))
smoke_targets = targets[:2].to(DEVICE)
smoke_losses = smoke_criterion(smoke_outputs, smoke_targets)
smoke_losses['total'].backward()
print('outputs:', {key: tuple(value.shape) for key, value in smoke_outputs.items()})
print('losses:', {key: float(value.detach().cpu()) for key, value in smoke_losses.items()})
print('SMOKE TEST: OK')
del smoke_model, smoke_outputs, smoke_losses

## 3. Khởi tạo model và optimizer

Giữ `USE_PRETRAINED=True` khi train thật trên UltraView.

In [ ]:
model = SpatialFireModel(pretrained=USE_PRETRAINED).to(DEVICE)
criterion = FireLoss(lambda_coord=5.0, lambda_heatmap=0.5)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, EPOCHS), eta_min=LEARNING_RATE / 100)
print('trainable parameters:', sum(p.numel() for p in model.parameters() if p.requires_grad))

## 4. Train

Checkpoint tốt nhất được chọn theo validation total loss. Metric localization được tính trên toàn epoch.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
best_val = float('inf')
history = []

for epoch in range(1, EPOCHS + 1):
    train_loss, train_metric = run_epoch(model, train_loader, criterion, DEVICE, optimizer=optimizer)
    val_loss, val_metric = run_epoch(model, val_loader, criterion, DEVICE)
    scheduler.step()
    row = {'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss, 'train_metric': train_metric, 'val_metric': val_metric}
    history.append(row)
    print(f"epoch={epoch:03d} train={train_loss['total']:.5f} val={val_loss['total']:.5f} MAE={val_metric['mae_px']:.2f}px PCK10={val_metric['pck10']:.3f} PCK25={val_metric['pck25']:.3f}")
    if val_loss['total'] < best_val:
        best_val = val_loss['total']
        save_checkpoint(OUTPUT_DIR / 'best_spatial.pth', model, optimizer, scheduler, epoch, best_val, split_info, TRAIN_ARGS)
    save_checkpoint(OUTPUT_DIR / 'last_spatial.pth', model, optimizer, scheduler, epoch, val_loss['total'], split_info, TRAIN_ARGS)
    (OUTPUT_DIR / 'history_notebook.json').write_text(json.dumps(history, indent=2), encoding='utf-8')

print('best validation loss:', best_val)
print('best checkpoint:', OUTPUT_DIR / 'best_spatial.pth')

## 5. Đánh giá test độc lập

In [ ]:
checkpoint_path = OUTPUT_DIR / 'best_spatial.pth'
if not checkpoint_path.exists():
    raise FileNotFoundError('Chưa có checkpoint. Hãy chạy cell Train trước.')
checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
if checkpoint.get('architecture') != ARCHITECTURE:
    raise RuntimeError(f"Sai architecture: {checkpoint.get('architecture')}")
model.load_state_dict(checkpoint['model'])
test_loss, test_metric = run_epoch(model, test_loader, criterion, DEVICE)
print('test_loss:', test_loss)
print('test_metric:', test_metric)
(OUTPUT_DIR / 'test_metrics_notebook.json').write_text(json.dumps({'loss': test_loss, 'metric': test_metric}, indent=2), encoding='utf-8')

## 6. Learning curve

In [ ]:
if not history:
    history_path = OUTPUT_DIR / 'history_notebook.json'
    if history_path.exists():
        history = json.loads(history_path.read_text(encoding='utf-8'))
if not history:
    raise RuntimeError('Chưa có history. Hãy chạy cell Train trước.')
epochs = [row['epoch'] for row in history]
train_total = [row['train_loss']['total'] for row in history]
val_total = [row['val_loss']['total'] for row in history]
val_mae = [row['val_metric']['mae_px'] for row in history]
val_pck10 = [row['val_metric']['pck10'] for row in history]
val_pck25 = [row['val_metric']['pck25'] for row in history]
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(epochs, train_total, label='train')
axes[0].plot(epochs, val_total, label='val')
axes[0].set_title('Total loss')
axes[0].legend()
axes[1].plot(epochs, val_mae, color='tab:orange')
axes[1].set_title('Validation MAE (px)')
axes[2].plot(epochs, val_pck10, label='PCK@10')
axes[2].plot(epochs, val_pck25, label='PCK@25')
axes[2].set_ylim(0, 1)
axes[2].legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'learning_curve_spatial.png', dpi=150)
plt.show()

## Ghi chú tích hợp

Checkpoint `best_spatial.pth` dùng `spatial_heatmap_v2`; adapter trong `fire_detector.py` đã hỗ trợ architecture này và không ghi đè checkpoint GAP cũ.